In [1]:
# 导入必要的库
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV

from matplotlib import rcParams

# 配置 matplotlib 使用字体
rcParams['font.sans-serif'] = ['Heiti TC']
rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

# 加载鸢尾花数据集
iris = load_iris()
X = iris.data  # 特征数据
y = iris.target  # 标签数据
feature_names = iris.feature_names
target_names = iris.target_names

# 数据集基本信息
print("数据集描述：")
print(iris.DESCR)

# 打印前5个样本
print("\n前5个样本：")
print(pd.DataFrame(X[:5], columns=feature_names))

# 打印每个类别的样本数量
print("\n每个类别的样本数量：")
print(pd.Series(y).value_counts())

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 特征标准化
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 创建随机森林分类器
rf_classifier = RandomForestClassifier(random_state=42)

# 训练模型
rf_classifier.fit(X_train, y_train)

# 在测试集上进行预测
y_pred = rf_classifier.predict(X_test)

# 模型评估
accuracy = accuracy_score(y_test, y_pred)
print(f"\n模型准确率：{accuracy:.4f}")

print("\n分类报告：")
print(classification_report(y_test, y_pred, target_names=target_names))

print("\n混淆矩阵：")
print(confusion_matrix(y_test, y_pred))

# 定义参数网格，指定需要搜索的超参数及其取值范围
param_grid = {
    # 'n_estimators'：随机森林中决策树的数量
    # 较多的树可以提高模型性能，但会增加计算成本
    # 这里尝试50棵、100棵和200棵决策树
    'n_estimators': [50, 100, 200],
    
    # 'max_depth'：决策树的最大深度
    # None表示树会一直生长直到节点中的样本都属于同一类别或达到其他停止条件
    # 较大的深度可能使模型过拟合，较小的深度可能使模型欠拟合
    # 这里尝试不限制深度，以及限制为10、20、30层的情况
    'max_depth': [None, 10, 20, 30],
    
    # 'min_samples_split'：节点分裂所需的最小样本数量
    # 较大的值可以防止对小样本节点的过度分裂，减少过拟合风险
    # 这里尝试2、5、10个样本作为分裂的最小要求
    'min_samples_split': [2, 5, 10]
}

# 创建网格搜索对象，用于自动搜索最优参数组合
# estimator=RandomForestClassifier(random_state=42)：使用随机森林分类器作为基础模型，设置随机种子确保结果可重复
# param_grid=param_grid：将定义的参数网格传入，告诉GridSearchCV需要搜索哪些参数
# cv=5：使用5折交叉验证，将训练数据分成5份，每次用4份训练，1份验证，全面评估模型性能
# n_jobs=-1：使用所有可用的CPU核心进行并行计算，加速网格搜索过程
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    n_jobs=-1
)

# 在训练集上进行网格搜索和交叉验证，自动尝试所有参数组合并找到最优解
grid_search.fit(X_train, y_train)

print(f"\n最优参数组合：{grid_search.best_params_}")
print(f"最佳交叉验证得分：{grid_search.best_score_:.4f}")

# 使用最优参数重新训练模型
best_rf = grid_search.best_estimator_
best_rf.fit(X_train, y_train)

# 在测试集上评估最优模型
y_pred_best = best_rf.predict(X_test)
accuracy_best = accuracy_score(y_test, y_pred_best)
print(f"\n最优模型在测试集上的准确率：{accuracy_best:.4f}")

# 应用示例：预测新样本
new_sample = [[5.1, 3.5, 1.4, 0.2]]  # 新样本数据
new_sample_scaled = scaler.transform(new_sample)
predicted_class = best_rf.predict(new_sample_scaled)
predicted_class_name = target_names[predicted_class][0]

print(f"\n新样本的预测类别：{predicted_class_name}")

数据集描述：
.. _iris_dataset:

Iris plants dataset
--------------------

**Data Set Characteristics:**

:Number of Instances: 150 (50 in each of three classes)
:Number of Attributes: 4 numeric, predictive attributes and the class
:Attribute Information:
    - sepal length in cm
    - sepal width in cm
    - petal length in cm
    - petal width in cm
    - class:
            - Iris-Setosa
            - Iris-Versicolour
            - Iris-Virginica

:Summary Statistics:

============== ==== ==== ======= ===== ====================
                Min  Max   Mean    SD   Class Correlation
============== ==== ==== ======= ===== ====================
sepal length:   4.3  7.9   5.84   0.83    0.7826
sepal width:    2.0  4.4   3.05   0.43   -0.4194
petal length:   1.0  6.9   3.76   1.76    0.9490  (high!)
petal width:    0.1  2.5   1.20   0.76    0.9565  (high!)
============== ==== ==== ======= ===== ====================

:Missing Attribute Values: None
:Class Distribution: 33.3% for each of 3 class